# RAG - Retrieval Augmented Generations    
* Is a patern that can improve the effecacy of a LLM application by leveraging custom data   
* Is done by retieving data/documents relevants to a question or task and providing them as context to augment the prompts to a LLM to improve generation.

#### RAG Use Cases    

* Q&A Chatbots   
* Search Augmentation   
* Content Criation and Summarization

#### Main concepts of RAG Workflow   
**1.  Index and Embed**: An embedding model used to creating vector representation of the documents and users queries.   
**2.  Vector store**: Specialized to store unstructured data indexed by vectros. Vectors can be sotred with a **vector DB**, **library** or **plugin**   
**3.  Retrieval**: Search stored vectors uing similairity search to efficiently retrieve relavant information   
**4.  Filtering & Reranking**: The process of selectting or raking retrieved documents before passing as context. Filtering can be **pre-, in-, post-query**    
**5.  Prompt Augmentation**: Prompt engineering workflow to enhace context via injections of data retrieved from Vector store    
**6.  Generation**: A LLM used for generating a response for the user's request. 

#### Benefits of RAG Architecture
* Up-to-date and accurate response    
* Reducing inaccurate response or hallucination    
* Damin-specific contextitualization   
* Efficiency and cost0effectiveness

In [0]:
%pip install mlflow==2.10.1 lxml==4.9.3 langchain==0.1.5 databricks-vectorsearch==0.22 cloudpickle==2.2.1 databricks-sdk==0.18.0 cloudpickle==2.2.1 pydantic==2.5.2 openpyxl
%pip install pip mlflow[databricks]==2.10.1

dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import random
import os
import uuid
import json
import io
import plotly.express as px
from langchain_community.chat_models import ChatDatabricks

from datetime import datetime
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

In [0]:
chat_endpoint = "databricks-llama-4-maverick" 
chat_model = ChatDatabricks(endpoint = chat_endpoint,
                            temperature = 0,
                            max_tokens = 8000) 

In [0]:
llm_summary = ChatDatabricks(endpoint="databricks-gpt-oss-120b")

In [0]:
parameters = {
"Persona":"Prefessor of AI enginerring, specilist in compound ai systems",
"Purpose":"Generate informations in the simple languages to facilitate the learning process",
"Parameters":"",
"Wish":"Summarize the text, presenting the main point without skip any importante topic",
"Patern":"Technical information in bullets format, but not too short bullets.",
"Audience": "AI Engineers students"
}

subject = "Compound AI system (AI Agents)"

system_prompt = f"""You are a prompt engineering specialist. You will create prompts following the p6 framework. You must follow the p6 framework in right way, by following correctly the instrucions. You will receive six parameters (Persona, Purpose, Parameters, Wish, Patern, Audience) in json format to be used in the framework.

p6 framework: 
The p6 framework define the prompt structure in six steps, as presented below.

1 Persona: Define whoe the AI must be or act.
	Exemples: 
	Act as a nutricionista...
	You are na sênior data analist...

2 Purpose: Be clair about what is the final objective.
	Exemples: 
	I need to create something...
	I need to sumarize the text...
	Help me solve the error in python...
		
3 Parameters: When available, provide relavante ata and contexto to be considered in the task.restrições) 

4 Wish: Tell to AI what exactly you need using use verbs in the present imperative.

5 Patern: Format and/or structure of the output

6 Audience: For who is the output for.


Generate a prompt to the subject: {subject} folowing the p6 framework with the parameters {parameters}

Yor answer must be addressed to a LLM model in json format, with the following structure:
{{"prompt": "Your prompt here"}}

Do not include any other information in your answer"""



In [0]:
text = """Alright, so that's what we're gonna do here. We're actually gonna jump into a demo. I'm gonna go jump into the notebooks. We're gonna go through each of these kind of components or intents and look at how do we define the APIs, interfaces, understand the dependencies and the components needed to go and build that, right. So in that we're gonna talk about how what it looks like.Run search, run summary, run, get context, run QA stage. And then we'll look at full multi endpoint car architecture where we have identified at the end of this kind of the interfaces, the flow as well as other dependencies that our system might rely on. Alright, let's jump into a demo. And this demo is really talking about.We just talked about in our slides, which is how to plan a compound AI system architecture. So we're gonna go over that architecture diagram and then dive a little bit deeper into how do we go and look at each of those components and stages. And then what would that kind of look at like the high level when you look at the code, things we wanna think about.When it comes to building out a system, in this demo, we'll talk about doing that in Python. We'll talk about really the goal here is to scope the functionalities and strengths of each of the components in the system and outline the structure and relationship of each of those components and address some of the technical challenges. So then when it comes to implementation, we know everything that's.So by the end of this demo, you should be able to apply a class architecture to the stages. During this decomposition phase, we'll be able to explain the conventions that map the stages to class methods, and then plan what methods and attributes are used to write a compound application and the various components in a compound app. Alright, so to start off, we're gonna run some of.Requirements. I've already ran this earlier, so it's basically gonna install some libraries. In this case, we're using Graphbiz to visualize the architecture to make it easier for you to see in HTML. And then we're gonna install a couple libraries right here. And then we define two functions here that are gonna be used to visualize that graphic that you saw on.The slides, it's just an HTML code that defines the graph that you see. So not gonna go into it here, but it's just all the HTML code that's used to generate that. So this is great, the HTML and then this is to highlight a particular stage in the HTML. So their helper methods help us visualize what we're doing.
"""

In [0]:
import json
prompt = chat_model.invoke(system_prompt).content
prompt = json.loads(prompt)["prompt"] + "\n" + f"text:{text}"
print(prompt)

In [0]:
llm_summary.invok